# Static workflow finetune

Tune `run_two_step_text_to_dataset` on one paper from `wopke_100`.

- **`workflow="facts"` (default):** SPO facts → optional facts→schema LLM (`use_llm_dataset_builder=True`). Knobs: `max_facts`, `document_excerpt_max_chars`, …
- **`workflow="label_then_direct"`:** Step 1 LLM returns **full paper + inline XML** (`LabeledPaperOutput.labeled_document` only — no tagging tool). Step 2: direct LLM prompts + completeness appendix. Knobs: `labeled_text_max_chars`, `label_step2_prompt_style`, `label_step2_include_tag_note`, `label_step2_maximize_completeness`.

Document input matches `evaluate_3`: **highlighted** text (`highlight_numbers_and_tables`).

**Outputs:** fact or pair counts, record count, normalized ROUGE-L vs ground truth; optional grid for the **facts** path only.

In [ ]:
import re
import sys
from typing import Any, Dict

sys.path.insert(0, "..")

import pandas as pd

from src.config import LLM_PROVIDER, get_model_name
from src.core.schema_factory import SchemaFactory
from src.experimentutils import (
    build_study_paper_mapping,
    evaluate_method_scores,
    highlight_numbers_and_tables,
    load_ground_truth,
    read_paper_text,
    save_extraction_results_with_timestamp,
)
from src.standards import METADATA_STANDARDS
from src.static_workflow import run_two_step_text_to_dataset

## Load ground truth and map Study# → paper file

In [ ]:
gt_df = load_ground_truth()
mapping = build_study_paper_mapping(gt_df)
print(f"Mapped {len(mapping)}/100 studies to paper files")

study_id = 1  # change to try another paper

if study_id not in mapping:
    raise ValueError(f"Study# {study_id} not in mapping. Try: {sorted(mapping.keys())[:20]} ...")

paper_path = mapping[study_id]
if paper_path.endswith(".pdf"):
    from src.experimentutils import convert_pdf_to_markdown

    paper_path = convert_pdf_to_markdown(paper_path)

paper_name = paper_path.split("/")[-1]
print(f"Study# {study_id} -> {paper_name}")

## Schema, GT rows, highlighted document (same convention as `evaluate_3`)

In [ ]:
standard_key = "wopke_100"
standard = METADATA_STANDARDS[standard_key]
n = study_id

factory = SchemaFactory()
wopke_field_names = list(factory._parse_schema_string(standard).keys())
n_fields = len(wopke_field_names)

provider_label = re.sub(r"[^A-Za-z0-9]+", "-", (LLM_PROVIDER or "unknown").strip()).strip("-")
model_label = re.sub(r"[^A-Za-z0-9]+", "-", get_model_name().strip()).strip("-")
file_tag = f"{provider_label}_{model_label}_{n_fields}fields"

gt_paper = gt_df[gt_df["Study#"] == study_id].reset_index(drop=True)
print(f"Standard: {standard_key} ({n_fields} fields)")
print(f"GT rows : {len(gt_paper)}")

raw_text = read_paper_text(paper_path)
highlighted = highlight_numbers_and_tables(raw_text)
print(f"Raw len: {len(raw_text):,} | Highlighted: {len(highlighted):,} chars")

gt_cols_set = set(gt_paper.columns)
shared_fields = [f for f in wopke_field_names if f in gt_cols_set]
print(f"Shared eval fields: {len(shared_fields)} / {len(wopke_field_names)}")

## Tunable parameters

Edit this dict (or the sweep lists below). `model_name` / `provider` as `None` use `src.config` defaults.

In [ ]:
WORKFLOW_KWARGS: Dict[str, Any] = {
    "workflow": "facts",  # or "label_then_direct" (labeller + XML + direct-LLM prompt)
    "max_facts": 200,
    "document_excerpt_max_chars": 56_000,
    "labeled_text_max_chars": 120_000,
    "label_step2_prompt_style": "direct_full",  # "direct_full" | "direct_simple"
    "label_step2_include_tag_note": True,
    "label_step2_maximize_completeness": True,
    "temperature": 0.0,
    "dataset_records_key": "yield_records",
    "record_class_name": "WopkeRecord",
    "output_class_name": "WopkeOutput",
    "pass_schema_hint_to_fact_extractor": True,
    "model_name": None,  # e.g. "google/gemini-2.5-flash"
    "provider": None,  # e.g. "google"
}

## Run static workflow (single configuration)

In [ ]:
_pass = {
    k: v
    for k, v in WORKFLOW_KWARGS.items()
    if not (k in ("model_name", "provider") and v is None)
}

out = run_two_step_text_to_dataset(
    text=highlighted,
    use_llm_dataset_builder=True,
    dataset_standard=standard,
    **_pass,
)

df_wf = out["dataset"]
n_rows = len(df_wf)
if out.get("workflow") == "label_then_direct":
    lt = out.get("labeled_text") or ""
    print(f"Labeled doc len : {len(lt):,} chars")
else:
    n_facts = len(out["facts_result"].facts)
    print(f"Facts extracted : {n_facts}")

print(f"Records built   : {n_rows} (GT has {len(gt_paper)})")
print(f"Validation OK   : {out['validation_ok']}")
if out["validation_issues"]:
    print("Issues:", out["validation_issues"])

df_wf.head(min(4, len(df_wf)))

### Inspect extracted facts (optional)

In [ ]:
(
    pd.DataFrame([f.model_dump() for f in out["facts_result"].facts]).head(15)
    if out.get("facts_result") is not None
    else "No SPO facts — use workflow=label_then_direct: see out['labeled_text'], out['label_step1']"
)

## Score vs ground truth (normalized ROUGE-L)

Same metric as `evaluate_3`: greedy record matching, per-field ROUGE-L, normalized by `len(gt_paper) * n_fields`.

In [ ]:
def score_workflow_df(ext_df: pd.DataFrame, label: str = "workflow") -> None:
    eval_cols = [c for c in shared_fields if c in ext_df.columns]
    if not eval_cols:
        print(f"[{label}] no overlapping columns with GT")
        return
    total_records = len(gt_paper)
    _, overall_norm, per_field_norm, n_matches = evaluate_method_scores(
        ext_df=ext_df,
        gt_df=gt_paper,
        shared_cols=eval_cols,
        total_records_for_denominator=total_records,
    )
    print(
        f"[{label}] extracted={len(ext_df)}, matched={n_matches}, "
        f"reference={total_records}, fields={len(eval_cols)} "
        f"-> normalized ROUGE-L = {overall_norm:.3f}"
    )
    return overall_norm, per_field_norm, n_matches


_ = score_workflow_df(df_wf, "static_workflow")

## Optional: small parameter sweep

Runs **multiple** API calls. Adjust `MAX_FACTS_GRID` and `DOC_CHARS_GRID` to your budget.

In [ ]:
RUN_SWEEP = False  # set True to run grid (many LLM calls)

MAX_FACTS_GRID = [120, 200, 280]
DOC_CHARS_GRID = [25_000, 40_000]

_base = {
    k: v
    for k, v in WORKFLOW_KWARGS.items()
    if k not in ("max_facts", "document_excerpt_max_chars") and v is not None
}

rows = []
if RUN_SWEEP:
    for mf in MAX_FACTS_GRID:
        for dc in DOC_CHARS_GRID:
            o = run_two_step_text_to_dataset(
                text=highlighted,
                use_llm_dataset_builder=True,
                dataset_standard=standard,
                max_facts=mf,
                document_excerpt_max_chars=dc,
                **_base,
            )
            dfo = o["dataset"]
            eval_cols = [c for c in shared_fields if c in dfo.columns]
            if not eval_cols:
                overall = float("nan")
                nm = 0
            else:
                _, overall, _, nm = evaluate_method_scores(
                    ext_df=dfo,
                    gt_df=gt_paper,
                    shared_cols=eval_cols,
                    total_records_for_denominator=len(gt_paper),
                )
            rows.append(
                {
                    "max_facts": mf,
                    "document_excerpt_max_chars": dc,
                    "n_facts": len(o["facts_result"].facts),
                    "n_records": len(dfo),
                    "n_matches": nm,
                    "normalized_rougeL": overall,
                }
            )
    sweep_df = pd.DataFrame(rows).sort_values("normalized_rougeL", ascending=False)
    sweep_df
else:
    print("Set RUN_SWEEP = True to run the grid.")

## Save best / current run (optional)

In [ ]:
SAVE = False  # set True to write CSV under outputs/

_out = globals().get("out")
if SAVE and _out and _out.get("schema_output") is not None:
    path = save_extraction_results_with_timestamp(
        results=_out["schema_output"],
        base_name=f"{n}_static_workflow_finetune_{file_tag}",
        records_key="yield_records",
        include_time=False,
    )
    print(f"Saved: {path}")
elif SAVE:
    print("Run the workflow cell first, or enable use_llm_dataset_builder (schema_output missing).")